# 影片前處理：SigLIP 2 Video Encoder
流程：影片 → 鏡頭切分 → 4 fps 擷取影格 → SigLIP 2 逐格編碼 → pooling → 每支影片一個向量
輸出：`features/video/{DATA_MODE}_{MODEL_TAG}.pt`，主 notebook 的 2.3 Video Encoder 直接讀取

**為什麼另開一本 notebook？**
1. 影片解碼與編碼最花時間，只需跑一次
2. 需要的套件（PySceneDetect、OpenCV）跟主模型無關
3. 快取以 `media_id` 命名，pilot 算過的影片，full 模式可以直接沿用

**和 MM-AU（§6.1）的對照**

| 步驟 | MM-AU | 本專案 |
|---|---|---|
| 鏡頭切分 | PySceneDetect | 同，ContentDetector |
| 影格擷取 | 4 fps | 同，每支上限 480 格 |
| 影格編碼 | CLIP ViT-B/32 | SigLIP 2 base-patch16-**naflex** |
| 影格 → 鏡頭 | average pool | 同 |
| 鏡頭 → 影片 | 無（鏡頭序列直接進 LSTM / Transformer） | **固定 32 格再 max-pool**（理由見第 10 節） |

# 0. 匯入與參數

| 參數 | 設定 | 為什麼 |
|---|---|---|
| `SAMPLE_FPS` | 4 | 照 MM-AU §6.1 |
| `MAX_FRAMES` | 480 | = 120 秒 × 4 fps。抽樣的 50 支影片最長 120 秒、75 百分位 70 秒。原本設 360，超過 90 秒的影片會改成均勻抽格，取樣密度就跟其他影片不一樣；改成 480 後，全部影片都維持 4 fps |
| `SIGLIP_MODEL` | naflex | 96% 是 9:16 直式。正方形輸入的版本，不是把畫面壓扁，就是把上下的字卡裁掉。NaFlex 保留原始長寬比，SigLIP 2 論文指出它在 OCR、文件類 benchmark 多半比較好 |
| `MAX_PATCHES` | 256 | 9:16 約 12×21 個 patch（約 192×336 px）。576 能看清更多畫面文字，但運算量約 2.5 倍，留作 ablation |
| `RESIZE_LONG` | 512 | 主流解析度 720×1280，讀進來先縮小，省記憶體；NaFlex 實際只用到約 336 px，不損失資訊 |
| `SCENE_THRESHOLD` / `MIN_SCENE_SEC` | 27 / 0.5 秒 | MM-AU 沒寫，用 PySceneDetect 預設值。Reels 剪接快，限制最短 0.5 秒，確保每個鏡頭至少抽得到一格 |
| `FIXED_K` | 32 | 每支影片均勻取 32 格再 max，消除「格數越多、最大值越高」的偏誤（見第 10 節） |
| `MAX_SHOTS` | 35 | 照 MM-AU；只影響存下的鏡頭序列，不影響 pooling 向量 |

> 論文沒有寫、自己決定的參數，在程式碼註解標了「← MM-AU 未給」或「← 自訂」

In [28]:
# 若尚未安裝：pip install "scenedetect[opencv]" "transformers>=4.49"
import warnings, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor
from scenedetect import open_video, SceneManager, ContentDetector

In [29]:
# ── 資料模式 ──
DATA_MODE = "pilot"        # "pilot" / "full"

ROOT      = Path("/home/yingtzudai/Multimodal_SPoD_Video")
SPLIT_CSV = {"pilot": ROOT / "data/split/pilot_posts.csv",
             "full":  ROOT / "data/split/post_split.csv"}[DATA_MODE]
DATA_ROOT = Path("/home/ftp_246/data_5/多模態2026_ig_top200")
VIDEO_DIR = DATA_ROOT / "Video_data_fullest/ig_vedios"          # {username}/{stem}.mp4
FEAT_DIR  = ROOT / "features"

SEED   = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16 if device.type == "cuda" else torch.float32
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── 鏡頭切分（MM-AU：PySceneDetect）──
SCENE_THRESHOLD = 27.0     # ContentDetector 預設值 ← MM-AU 未給
MIN_SCENE_SEC   = 0.5      # 最短鏡頭長度          ← MM-AU 未給

# ── 影格擷取（MM-AU：4 fps）──
SAMPLE_FPS  = 4
MAX_FRAMES  = 480          # 超過 90 秒的影片改為均勻抽 360 格 ← 自訂
RESIZE_LONG = 512          # 讀進來先把長邊縮到 512，省記憶體（NaFlex 實際只用到約 336px）

# ── SigLIP 2 ──
SIGLIP_MODEL = "google/siglip2-base-patch16-naflex"   # ablation："google/siglip2-base-patch16-224"
MAX_PATCHES  = 256         # NaFlex 序列長度；9:16 約 12×21 個 patch
FRAME_BATCH  = 32
PROC_KW = {"max_num_patches": MAX_PATCHES} if "naflex" in SIGLIP_MODEL else {}

# ── Pooling / 序列 ──
FIXED_K   = 32             # 每支影片均勻取 K 格再 max（消除格數造成的極值效應）
POOLS     = ["max_k", "mean_shot", "mean_frame", "max"]   # 第一個是預設，其餘為 ablation
MAX_SHOTS = 35

# ── 存檔 ──
MODEL_TAG = (SIGLIP_MODEL.split("/")[-1]
             + (f"_p{MAX_PATCHES}" if PROC_KW else "") + f"_fps{SAMPLE_FPS}")
CACHE_DIR = FEAT_DIR / "video" / "cache" / MODEL_TAG   # 每支影片一個檔，pilot / full 共用
OUT_PT    = FEAT_DIR / "video" / f"{DATA_MODE}_{MODEL_TAG}.pt"
SAVE_FRAME_EMB = True      # 保留逐格 embedding，之後換 pooling 不用重算
RECOMPUTE      = False
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(device, "|", MODEL_TAG)

cuda | siglip2-base-patch16-naflex_p256_fps4


# 1. 貼文清單與影片路徑

In [30]:
posts = pd.read_csv(SPLIT_CSV, dtype={"media_id": str, "stem": str})
posts["media_id"]   = posts["media_id"].str.strip()
posts["video_path"] = [VIDEO_DIR / u / f"{s}.mp4" for u, s in zip(posts["username"], posts["stem"])]
posts["has_video"]  = posts["video_path"].map(Path.exists)

print(posts.shape, "| 找得到影片的比例：", posts["has_video"].mean().round(3))
print(posts.loc[~posts["has_video"], ["username", "stem"]].head())

(299, 9) | 找得到影片的比例： 1.0
Empty DataFrame
Columns: [username, stem]
Index: []


# 2. 影片格式檢查
只讀檔頭、不解碼，很快。用來決定 `MAX_FRAMES`、`RESIZE_LONG`。

**抽樣結果（n = 50）**
- fps：23.98–30，大多是 30
- 解析度：主流 720×1280，最小 360×640
- 長度：中位數 44.7 秒，75 百分位 70.5 秒，最長 120.3 秒
- 直式比例 96%；4 fps 下平均每支 198 格

**為什麼不排除 outlier（橫式、極短、很長的影片）？**
1. **避免選擇偏誤**：SPoD 要對所有貼文排序，實際使用時一定會遇到這些影片
2. **格式本身可能是訊號**：橫式可能是品牌提供的廣告素材；長度也跟業配有關（業配中位數 59.9 秒，非業配 37.1 秒）
3. **流程本身能處理**：NaFlex 不會讓橫式變形；最短的影片也有約 35 格；pooling 後每支都是同樣維度的向量

真正排除的只有技術上壞掉的檔案，而且不刪資料，只標成 `video_ok = False`，當作「缺少影片模態」。

In [31]:
def probe(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    n   = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    return {"fps": fps, "n_frames": n, "width": w, "height": h,
            "duration": n / fps if fps else np.nan}

sample = posts[posts["has_video"]].sample(min(50, posts["has_video"].sum()), random_state=SEED)
info = pd.DataFrame([probe(p) or {} for p in sample["video_path"]])
print(info.describe().T.round(2))
print("直式比例：", (info["height"] > info["width"]).mean().round(3))
print("4 fps 下平均每支影格數：", (info["duration"] * SAMPLE_FPS).mean().round(0))

          count     mean     std     min      25%      50%      75%      max
fps        50.0    29.76    1.19   23.98    30.00    30.00    30.00    30.00
n_frames   50.0  1476.32  886.87  261.00   725.25  1301.00  2115.50  3610.00
width      50.0   716.80  108.07  360.00   720.00   720.00   720.00  1280.00
height     50.0  1232.00  164.83  640.00  1280.00  1280.00  1280.00  1280.00
duration   50.0    49.45   29.44    8.70    24.18    44.66    70.52   120.33
直式比例： 0.96
4 fps 下平均每支影格數： 198.0


In [32]:
# 全部影片都讀一次檔頭，結果併回 posts
info_all = pd.DataFrame([probe(p) if ok else {} for p, ok in zip(posts["video_path"], posts["has_video"])],
                        index=posts.index)
posts = posts.join(info_all)
posts["orient"] = np.where(posts["height"] > posts["width"], "portrait",
                  np.where(posts["height"] < posts["width"], "landscape", "square"))
posts["dur_bin"] = pd.cut(posts["duration"], [0, 15, 30, 60, 90, np.inf],
                          labels=["<15s", "15-30s", "30-60s", "60-90s", ">90s"])

print("讀不到 / 長度 < 1 秒：", int(((posts["duration"] < 1) | posts["duration"].isna()).sum()), "支")
print("\n各方向的業配比例")
print(posts.groupby("orient")["label"].agg(["mean", "count"]).round(3))
print("\n各長度區間的業配比例")
print(posts.groupby("dur_bin", observed=True)["label"].agg(["mean", "count"]).round(3))
print("\n超過 MAX_FRAMES 上限、會改為均勻抽格的比例：",
      (posts["duration"] * SAMPLE_FPS > MAX_FRAMES).mean().round(3))

讀不到 / 長度 < 1 秒： 0 支

各方向的業配比例
            mean  count
orient                 
landscape  0.600      5
portrait   0.454    293
square     0.000      1

各長度區間的業配比例
          mean  count
dur_bin              
<15s     0.211     38
15-30s   0.227     44
30-60s   0.419    124
60-90s   0.750     64
>90s     0.621     29

超過 MAX_FRAMES 上限、會改為均勻抽格的比例： 0.047


# 3. 鏡頭切分（PySceneDetect）
- 用 `ContentDetector`：比較相鄰畫面的色彩差異，差異超過門檻就算換鏡頭
- 一鏡到底的影片 → 整支當成一個鏡頭

**檢查結果（pilot）**：鏡頭數中位數 13、最多 122；每秒最多 1.19 個鏡頭，沒有發現閃光或字卡被大量誤判，門檻不用調整。

In [33]:
def detect_shots(path, fps, duration):
    video = open_video(str(path))
    sm = SceneManager()
    sm.add_detector(ContentDetector(threshold=SCENE_THRESHOLD,
                                    min_scene_len=max(1, int(MIN_SCENE_SEC * fps))))
    sm.detect_scenes(video, show_progress=False)
    scenes = sm.get_scene_list(start_in_scene=True)
    shots = [(s.get_seconds(), e.get_seconds()) for s, e in scenes]
    return shots or [(0.0, duration)]          # 一鏡到底 → 整支當一個鏡頭

# 抽 3 支看切得合不合理（可對照影片手動檢查）
for p in sample["video_path"].head(3):
    i = probe(p)
    s = detect_shots(p, i["fps"], i["duration"])
    print(p.name, f"{i['duration']:.1f}s →", len(s), "個鏡頭", [round(a, 1) for a, _ in s][:10])

/tmp/ipykernel_2365750/1479378777.py:8: DeprecationWarning: get_seconds() is deprecated, use the `seconds` property instead.
  shots = [(s.get_seconds(), e.get_seconds()) for s, e in scenes]


thetiffanychen-20251228211256-1216903180549089.mp4 94.6s → 50 個鏡頭 [0.0, 5.4, 6.6, 7.3, 7.9, 8.6, 13.4, 15.4, 16.9, 18.7]
1_shiuan_0-20250908004431-1635094370996640.mp4 51.7s → 13 個鏡頭 [0.0, 10.3, 16.4, 18.1, 34.6, 35.9, 38.6, 44.3, 46.5, 47.7]
zoebitalk-20250630230332-1492024018951094.mp4 13.8s → 11 個鏡頭 [0.0, 5.6, 6.4, 7.1, 7.7, 8.5, 9.4, 10.1, 10.7, 11.8]


# 4. 影格擷取（4 fps）
- 依序讀每一格，只解碼需要的那幾格（`grab` / `retrieve`），比用 `CAP_PROP_POS_MSEC` 跳著讀更準
- 用影片自己的 fps 換算時間，23.98 fps 這種非整數也能算對
- 超過 480 格（約 2 分鐘）才改成均勻抽格

In [34]:
def sample_frames(path, fps, n_frames):
    duration = n_frames / fps
    times = np.arange(0, duration, 1 / SAMPLE_FPS)
    if len(times) > MAX_FRAMES:
        times = np.linspace(0, duration, MAX_FRAMES, endpoint=False)
    targets = set(np.minimum(np.round(times * fps).astype(int), int(n_frames) - 1).tolist())
    last = max(targets)

    cap, frames, ts, i = cv2.VideoCapture(str(path)), [], [], 0
    while i <= last and cap.grab():
        if i in targets:
            ok, img = cap.retrieve()
            if ok:
                h, w = img.shape[:2]
                s = RESIZE_LONG / max(h, w)
                if s < 1:
                    img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
                frames.append(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
                ts.append(i / fps)
        i += 1
    cap.release()
    return frames, np.array(ts)

# 5. SigLIP 2 影格 embedding
- `get_image_features` 拿到的是 MAP head（attention pooling）的輸出。SigLIP 沒有 [CLS]，這就是整張圖的代表向量，base 版 768 維
- 每格先做 **L2 normalize**：SigLIP 是在 cosine 空間訓練的，先 normalize 再平均或取最大，比較合理
- 對照組：`google/siglip2-base-patch16-224`（正方形輸入）。SigLIP 2 論文指出，B 尺寸在自然影像上標準版比 NaFlex 好，所以留作 ablation

In [35]:
processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
siglip = AutoModel.from_pretrained(SIGLIP_MODEL, torch_dtype=DTYPE).to(device).eval()

@torch.no_grad()
def encode_frames(frames):
    out = []
    for i in range(0, len(frames), FRAME_BATCH):
        inputs = processor(images=frames[i:i + FRAME_BATCH], return_tensors="pt", **PROC_KW).to(device)
        inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)
        e = siglip.get_image_features(**inputs)
        if not torch.is_tensor(e):              # 新版 transformers 可能回傳 output 物件
            e = e.pooler_output
        out.append(F.normalize(e.float(), dim=-1).cpu())
    return torch.cat(out)                       # (影格數, d)；base = 768

EMB_DIM = siglip.config.vision_config.hidden_size
print("EMB_DIM:", EMB_DIM)

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

EMB_DIM: 768


# 6. Pooling：影格 → 鏡頭 → 影片

| 名稱 | 做法 | 角色 |
|---|---|---|
| `max_k` | 均勻取 32 格，每一維取最大值 | **預設** |
| `mean_shot` | 先平均成鏡頭，再平均所有鏡頭（每個鏡頭等權） | ablation |
| `mean_frame` | 所有影格直接平均（等於依時長加權） | ablation |
| `max` | 所有影格取最大值（SPoD 的 image max-pool） | ablation：會混進影片長度 |
| `shots` | 鏡頭序列，最多 35 個 | 之後做 attention pooling 用 |

選擇過程見第 10 節。

In [36]:
def pool_video(frame_emb, ts, shots):
    shot_embs = []
    for s, e in shots:
        m = torch.from_numpy((ts >= s) & (ts < e))
        if m.any():                             # 太短、沒抽到影格的鏡頭跳過
            shot_embs.append(frame_emb[m].mean(0))
    S = torch.stack(shot_embs) if shot_embs else frame_emb.mean(0, keepdim=True)
    k_idx = torch.linspace(0, len(frame_emb) - 1, FIXED_K).round().long()
    return {
        "shots":      S,
        "max_k":      frame_emb[k_idx].max(0).values,   # 預設：固定 K 格再 max
        "mean_shot":  S.mean(0),
        "mean_frame": frame_emb.mean(0),
        "max":        frame_emb.max(0).values,          # ablation：受格數影響
    }

# 7. 批次執行（可中斷續跑）
- 每支影片存成 `cache/{MODEL_TAG}/{media_id}.pt`，已經存在就跳過，中斷後重跑會接著做
- 逐格 embedding 也一起存（`SAVE_FRAME_EMB`）：之後換 pooling 方法不用重新編碼，第 10 節的固定格數檢驗就是靠這個
- 單支出錯不會讓整批停下來，狀態記錄在 `*_log.csv`
- 相鄰影格平均 cosine > 0.99 → 標 `static`（例如單張圖片做成的影片），之後再決定要不要當成缺模態

**Pilot 結果**：299 / 299 支成功編碼

In [37]:
def process_video(row):
    out = CACHE_DIR / f"{row.media_id}.pt"
    if out.exists() and not RECOMPUTE:
        return "cached"
    if not Path(row.video_path).exists():
        return "missing"
    info = probe(row.video_path)
    if info is None or not info["fps"] or info["n_frames"] < 1:
        return "unreadable"
    shots = detect_shots(row.video_path, info["fps"], info["duration"])
    frames, ts = sample_frames(row.video_path, info["fps"], info["n_frames"])
    if not frames:
        return "no_frames"
    emb = encode_frames(frames)
    # 幾乎靜止（相鄰影格平均 cosine > 0.99，例如把單張圖片做成影片）→ 一樣存檔，但做標記
    static = len(emb) > 1 and float((emb[1:] * emb[:-1]).sum(-1).mean()) > 0.99
    pooled = pool_video(emb, ts, shots)
    torch.save({"media_id": row.media_id, "model": SIGLIP_MODEL, "info": info,
                "ts": ts, "shots_sec": shots,
                "frame_emb": emb.half() if SAVE_FRAME_EMB else None,
                **{k: v.half() for k, v in pooled.items()}}, out)
    return "ok"

log = []
for row in tqdm(posts.itertuples(), total=len(posts), desc="video"):
    try:
        status = process_video(row)
    except Exception as ex:
        status = f"error: {type(ex).__name__}: {ex}"
    log.append((row.media_id, status))

log = pd.DataFrame(log, columns=["media_id", "status"])
print(log["status"].str.split(":").str[0].value_counts())
log.to_csv(CACHE_DIR.parent / f"{DATA_MODE}_{MODEL_TAG}_log.csv", index=False)

video:   0%|          | 0/299 [00:00<?, ?it/s]

status
cached    299
Name: count, dtype: int64


# 8. 彙整存檔
輸出格式比照 text embedding：用 `media_id` 對齊，主 notebook 依 `posts` 的順序取出。

| 欄位 | 內容 |
|---|---|
| `X_V[pool]` | (貼文數, 768)，各種 pooling 的影片向量 |
| `shots`, `shot_mask` | (貼文數, 35, 768) 鏡頭序列與 mask |
| `video_ok` | 哪些貼文有成功編碼的影片，aspect attention 處理缺模態時用 |
| `config` | 產生這份檔案的所有參數 |

11.7% 的影片超過 35 個鏡頭，`shots` 只留前 35 個；pooling 向量是用全部鏡頭算的，不受影響。之後做 attention pooling 時，建議改成均勻抽 35 個鏡頭。

In [38]:
N = len(posts)
X_V       = {p: torch.zeros(N, EMB_DIM) for p in POOLS}
shots_pad = torch.zeros(N, MAX_SHOTS, EMB_DIM, dtype=torch.float16)
shot_mask = torch.zeros(N, MAX_SHOTS, dtype=torch.bool)
video_ok  = torch.zeros(N, dtype=torch.bool)
n_shots   = np.zeros(N, dtype=int)

for i, m in enumerate(posts["media_id"]):
    f = CACHE_DIR / f"{m}.pt"
    if not f.exists():
        continue
    c = torch.load(f, weights_only=False)
    for p in POOLS:
        if p in c:
            X_V[p][i] = c[p].float()
        elif p == "max_k":                      # 舊快取沒有 max_k，用逐格 embedding 補算
            fe = c["frame_emb"].float()
            X_V[p][i] = fe[torch.linspace(0, len(fe) - 1, FIXED_K).round().long()].max(0).values
    S = c["shots"][:MAX_SHOTS]
    shots_pad[i, :len(S)] = S
    shot_mask[i, :len(S)] = True
    n_shots[i] = len(c["shots"])
    video_ok[i] = True

torch.save({"media_id": posts["media_id"].tolist(), "X_V": X_V,
            "shots": shots_pad, "shot_mask": shot_mask, "video_ok": video_ok,
            "config": {"model": SIGLIP_MODEL, "max_patches": MAX_PATCHES, "fps": SAMPLE_FPS,
                       "scene_threshold": SCENE_THRESHOLD, "min_scene_sec": MIN_SCENE_SEC,
                       "max_shots": MAX_SHOTS, "fixed_k": FIXED_K}}, OUT_PT)

print("已存檔：", OUT_PT)
print("成功編碼：", int(video_ok.sum()), "/", N)
print("鏡頭數分佈：", pd.Series(n_shots[video_ok.numpy()]).describe().round(1).to_dict())
print("超過 35 個鏡頭（會被截斷）的比例：", (n_shots > MAX_SHOTS).mean().round(3))

已存檔： /home/yingtzudai/Multimodal_SPoD_Video/features/video/pilot_siglip2-base-patch16-naflex_p256_fps4.pt
成功編碼： 299 / 299
鏡頭數分佈： {'count': 299.0, 'mean': 17.1, 'std': 16.2, 'min': 1.0, '25%': 4.0, '50%': 13.0, '75%': 25.0, 'max': 122.0}
超過 35 個鏡頭（會被截斷）的比例： 0.117


# 9. 快速檢查：影片向量有沒有業配訊號
用 logistic regression 做 linear probe，5-fold 交叉驗證。

**為什麼要按網紅分組（StratifiedGroupKFold）？**
同一位網紅的影片風格很像。如果同一位網紅的影片同時出現在訓練和測試裡，模型可能只是認出「這是常接業配的某某某」，分數會被高估。分組後，測試的網紅一定是模型沒看過的。

隨機基準 AP = 業配比例 = 136 / 299 ≈ 0.455

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import average_precision_score

ok = video_ok.numpy()
yv, g = posts["label"].values[ok], posts["username"].values[ok]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
print("隨機基準 AP ≈", yv.mean().round(3))
for p in POOLS:
    prob = cross_val_predict(LogisticRegression(max_iter=2000), X_V[p][ok].numpy(), yv,
                             cv=cv, groups=g, method="predict_proba")[:, 1]
    print(f"{p:10s} AP = {average_precision_score(yv, prob):.3f}")

隨機基準 AP ≈ 0.455
max_k      AP = 0.774
mean_shot  AP = 0.693
mean_frame AP = 0.686
max        AP = 0.775


# 10. 穩健性檢查：pooling 方法怎麼選出來的
每個設定用 10 個隨機種子重複，報告 AP 的平均 ± 標準差。

**第一、二輪：`max` 最好，但跟長度高度相關**
- `max` 0.785，遠高於 `mean_shot` 0.651、`mean_frame` 0.659
- 但只用影片長度就有 0.649；業配中位數 59.9 秒、非業配 37.1 秒
- `max` 向量的整體高度和 log(長度) 相關 0.686：格數越多，最大值必然越大
- 扣除長度效應後，`max` 掉到 0.553
→ 懷疑 `max` 只是在間接記錄影片長度

**第三輪：明確給出長度後，`max` 仍然領先**
- 長度 + mean_shot 0.657、長度 + max 0.783
→ `max` 有「線性長度」以外的資訊，可能是 (a) 非線性的長度效應，或 (b) 真的有關鍵畫面

**第四輪：分開兩種解釋**
- 每支都固定抽 32 格 → 格數造成的效應消失；長度改用 spline → 任何形狀都學得到
- 非線性長度單獨只有 0.625 → 排除 (a)
- `max_32` − `mean_32` = +0.086 ± 0.025，10 次全勝 → 支持 (b)
- 給了非線性長度後，`max_32` 仍比 `mean_32` 多 0.075，10 次全勝

**決定**：預設用 `max_k`（K = 32）。保留 SPoD「任何一張畫面出現產品就算」的 max-pool 邏輯，又不受影片長度影響。

**影片對文字有沒有幫助？**
- 跟「text + 長度」相比，加上 `max_32` 多 0.043、加上 `mean_32` 多 0.036，都是 10 次全勝 → 值得整合
- 有文字時，max 和 mean 的差距縮小（0.849 對 0.841），推測關鍵畫面的資訊和 caption 有部分重疊
- 長度單獨當 feature 只多約 0.006 → 不另設 aspect，之後當作 graph 的 post node feature

**限制**
- linear probe 不等於最終的 SPoD 模型
- 10 次重複用的是同一批資料，「10 次全勝」不是正式的顯著性檢定
- K = 32 是暫定值，正式實驗比較 16 / 32 / 64
- 業配和長度的關係要在 full 資料上再確認一次

## 10.1 重複交叉驗證，並納入影片長度與文字作對照
每個 seed 都換一種切法、重跑 10 次，看平均和標準差。另外加入幾組對照：只用影片長度、把長度效應從 max 扣掉、只用文字，以及文字加影片

In [40]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 影片長度與鏡頭數（從快取讀）
dur = np.array([torch.load(CACHE_DIR / f"{m}.pt", weights_only=False)["info"]["duration"]
                for m in posts["media_id"]])
ok  = video_ok.numpy()
yv, g = posts["label"].values[ok], posts["username"].values[ok]
log_dur = np.log(dur[ok])[:, None]

# Text embedding（與主 notebook 同一份）
t = torch.load(FEAT_DIR / "text" / "pilot_ckiplab_bert-base-chinese_masked.pt", weights_only=False)
tpos = {m: i for i, m in enumerate(t["media_id"])}
X_T = t["X_T"][[tpos[m] for m in posts["media_id"]]].numpy()[ok]

# 把 max 向量中可由「影片長度」線性解釋的部分扣掉（不使用 label，不會洩漏）
def residualize(X, z):
    Z = np.c_[np.ones(len(z)), z]
    return X - Z @ np.linalg.lstsq(Z, X, rcond=None)[0]

V = {p: X_V[p][ok].numpy() for p in POOLS}
candidates = {
    "長度 only":          np.c_[log_dur, np.log(n_shots[ok])[:, None]],
    "mean_shot":          V["mean_shot"],
    "mean_frame":         V["mean_frame"],
    "max":                V["max"],
    "max（扣除長度效應）": residualize(V["max"], log_dur),
    "text":               X_T,
    "text + mean_shot":   np.c_[X_T, V["mean_shot"]],
    "text + max":         np.c_[X_T, V["max"]],
}

def grouped_ap(X, seeds=range(10)):
    aps = []
    for s in seeds:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
        prob = cross_val_predict(clf, X, yv, cv=cv, groups=g, method="predict_proba")[:, 1]
        aps.append(average_precision_score(yv, prob))
    return np.mean(aps), np.std(aps)

print(f"隨機基準 AP ≈ {yv.mean():.3f}　｜　網紅數：{len(np.unique(g))}")
res = pd.DataFrame({k: grouped_ap(X) for k, X in candidates.items()}, index=["AP_mean", "AP_sd"]).T
print(res.round(3))

隨機基準 AP ≈ 0.455　｜　網紅數：99
                  AP_mean  AP_sd
長度 only             0.649  0.015
mean_shot           0.651  0.018
mean_frame          0.659  0.021
max                 0.785  0.017
max（扣除長度效應）         0.553  0.016
text                0.800  0.015
text + mean_shot    0.833  0.014
text + max          0.857  0.011


## 10.2 max與影片長度的關聯

In [41]:
mx_level = V["max"].mean(1)          # max 向量的整體高度
print("max 向量平均值 vs log(長度) 相關：", np.corrcoef(mx_level, log_dur[:, 0])[0, 1].round(3))
print("業配 / 非業配的影片長度中位數（秒）：",
      np.median(dur[ok][yv == 1]).round(1), "/", np.median(dur[ok][yv == 0]).round(1))

max 向量平均值 vs log(長度) 相關： 0.686
業配 / 非業配的影片長度中位數（秒）： 59.9 / 37.1


## 10.3 鏡頭數異常的影片

In [42]:
chk = posts.loc[ok, ["username", "stem", "label"]].assign(
    n_shots=n_shots[ok], duration=dur[ok].round(1))
chk["shots_per_sec"] = (chk["n_shots"] / chk["duration"]).round(2)
print(chk.sort_values("shots_per_sec", ascending=False).head(8).to_string())

           username                                            stem  label  n_shots  duration  shots_per_sec
204        japanuts        japanuts-20251228201320-2331979960639984      1       45      37.8           1.19
243        japanuts       japanuts-20251114225239-25897867413199158      1       16      15.7           1.02
123       hooleefun       hooleefun-20250727011908-1788470765402591      1       28      31.5           0.89
287          1105ya          1105ya-20250909075605-1937957040180651      0       10      11.4           0.88
275          khshu_           khshu_-20250610213234-930772389898260      1       45      53.6           0.84
0         ala780705       ala780705-20251214122010-2615877615465778      0       29      35.9           0.81
164       zoebitalk       zoebitalk-20250630230332-1492024018951094      0       11      13.8           0.80
70   ladyflavor_cui  ladyflavor_cui-20250726210517-1410621970514561      0       46      58.0           0.79


## 10.4 扣除長度後，影片內容的淨貢獻

In [43]:
L = candidates["長度 only"]
more = {
    "mean_shot（扣除長度效應）": residualize(V["mean_shot"], log_dur),   # 與 max 的同條件對照
    "長度 + mean_shot":          np.c_[L, V["mean_shot"]],
    "長度 + max":                np.c_[L, V["max"]],
    "text + 長度":               np.c_[X_T, L],
    "text + 長度 + mean_shot":   np.c_[X_T, L, V["mean_shot"]],
    "text + 長度 + max":         np.c_[X_T, L, V["max"]],
}
res2 = pd.DataFrame({k: grouped_ap(X) for k, X in more.items()}, index=["AP_mean", "AP_sd"]).T
print(res2.round(3))

                       AP_mean  AP_sd
mean_shot（扣除長度效應）        0.504  0.014
長度 + mean_shot           0.657  0.017
長度 + max                 0.783  0.017
text + 長度                0.808  0.015
text + 長度 + mean_shot    0.833  0.014
text + 長度 + max          0.856  0.011


## 10.5 固定格數pooling與非線性長度

In [44]:
from sklearn.preprocessing import SplineTransformer

# 非線性長度：log(秒數) 的 spline 展開 + log(鏡頭數)（不使用 label，不會洩漏）
L_nl = np.c_[SplineTransformer(n_knots=6, degree=3).fit_transform(log_dur),
             np.log(n_shots[ok])[:, None]]

# 固定格數 pooling：每支影片均勻取 K 格，消除「格數越多、最大值越高」的效應
K = 32
def fixed_k_pool(media_id, k=K):
    fe = torch.load(CACHE_DIR / f"{media_id}.pt", weights_only=False)["frame_emb"].float()
    sub = fe[np.linspace(0, len(fe) - 1, k).round().astype(int)]
    return sub.mean(0).numpy(), sub.max(0).values.numpy()

_p = [fixed_k_pool(m) for m in tqdm(posts["media_id"][ok], desc=f"fixed-{K}")]
mean_k = np.stack([a for a, _ in _p])
max_k  = np.stack([b for _, b in _p])
print("max_K 平均值 vs log(長度) 相關：", np.corrcoef(max_k.mean(1), log_dur[:, 0])[0, 1].round(3))

def ap_runs(X, seeds=range(10)):
    out = []
    for s in seeds:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
        prob = cross_val_predict(clf, X, yv, cv=cv, groups=g, method="predict_proba")[:, 1]
        out.append(average_precision_score(yv, prob))
    return np.array(out)

runs = {
    "長度(非線性)":           L_nl,
    "max（原本）":            V["max"],
    f"mean_{K}":              mean_k,
    f"max_{K}":               max_k,
    "長度(非線性) + max":      np.c_[L_nl, V["max"]],
    f"長度(非線性) + mean_{K}": np.c_[L_nl, mean_k],
    f"長度(非線性) + max_{K}":  np.c_[L_nl, max_k],
    "text + 長度(非線性)":     np.c_[X_T, L_nl],
    f"text + 長度(非線性) + mean_{K}": np.c_[X_T, L_nl, mean_k],
    f"text + 長度(非線性) + max_{K}":  np.c_[X_T, L_nl, max_k],
    "text + 長度(非線性) + max":       np.c_[X_T, L_nl, V["max"]],
}
R = {k: ap_runs(X) for k, X in tqdm(runs.items(), desc="CV")}
print(pd.DataFrame({k: [v.mean(), v.std()] for k, v in R.items()},
                   index=["AP_mean", "AP_sd"]).T.round(3))

# 成對比較：同樣 10 種切法下的差值
def paired(a, b):
    d = R[a] - R[b]
    return f"{a}  −  {b}:  Δ = {d.mean():+.3f} ± {d.std():.3f}｜10 次中贏 {int((d > 0).sum())} 次"

print()
for a, b in [(f"max_{K}", f"mean_{K}"),
             (f"長度(非線性) + max_{K}", "長度(非線性)"),
             (f"長度(非線性) + max_{K}", f"長度(非線性) + mean_{K}"),
             (f"text + 長度(非線性) + mean_{K}", "text + 長度(非線性)"),
             (f"text + 長度(非線性) + max_{K}", "text + 長度(非線性)")]:
    print(paired(a, b))

fixed-32:   0%|          | 0/299 [00:00<?, ?it/s]

max_K 平均值 vs log(長度) 相關： 0.504


CV:   0%|          | 0/11 [00:00<?, ?it/s]

                          AP_mean  AP_sd
長度(非線性)                     0.625  0.013
max（原本）                     0.785  0.017
mean_32                     0.655  0.021
max_32                      0.741  0.021
長度(非線性) + max               0.776  0.019
長度(非線性) + mean_32           0.659  0.027
長度(非線性) + max_32            0.734  0.020
text + 長度(非線性)              0.806  0.019
text + 長度(非線性) + mean_32    0.841  0.012
text + 長度(非線性) + max_32     0.849  0.011
text + 長度(非線性) + max        0.855  0.011

max_32  −  mean_32:  Δ = +0.086 ± 0.025｜10 次中贏 10 次
長度(非線性) + max_32  −  長度(非線性):  Δ = +0.109 ± 0.015｜10 次中贏 10 次
長度(非線性) + max_32  −  長度(非線性) + mean_32:  Δ = +0.075 ± 0.024｜10 次中贏 10 次
text + 長度(非線性) + mean_32  −  text + 長度(非線性):  Δ = +0.036 ± 0.015｜10 次中贏 10 次
text + 長度(非線性) + max_32  −  text + 長度(非線性):  Δ = +0.043 ± 0.015｜10 次中贏 10 次
